# Missingness Diagnostics

**Capstone Project — Moody's Ratings**  
*Pipeline step 2 of 4: Data quality assessment*

This notebook examines patterns of missing data in the master panel before imputation. It produces:
1. Variable-level missingness (which indicators have gaps, and how large)
2. Country-level missingness (which countries have poor coverage)
3. Temporal patterns (are earlier or later years more complete)
4. Correlation diagnostics (pairwise correlations using complete observations)
5. Comparison of summary statistics between high-missingness and low-missingness countries

**Input:** `master_data_wide.csv` (from Step 1).

**Sample selection:** Following the report methodology, we restrict the sample to countries where extractive resource rents (total NR rents minus forest rents) exceeded 1% of GDP in 1995, with Gulf states guaranteed inclusion regardless of threshold, and that were not classified as high-income by the World Bank in 1995. Non-sovereign territories (Hong Kong, etc.) are also excluded. This isolates the set of resource-dependent developing economies that form the basis of the analysis.

---

## 0. Setup and Data Loading

In [1]:
import os
os.makedirs("intermediary", exist_ok=True)

import pandas as pd
import numpy as np

In [2]:
# Load master data
master = pd.read_csv("intermediary/master_data_wide.csv")

# Drop index artefact if present
if "Unnamed: 0" in master.columns:
    master.drop(columns="Unnamed: 0", inplace=True)

# ── Sample selection (adjusted rents: total NR rents minus forest) ──
# The World Bank total NR rents indicator (NY.GDP.TOTL.RT.ZS) includes forest
# rents, so countries can qualify as "resource-rich" on forestry alone. We
# subtract forest rents to isolate extractive (oil, gas, mineral) rents.
RENT_COL   = "Total natural resources rents (% of GDP)"
FOREST_COL = "Forest rents (% of GDP)"
THRESHOLD  = 1.0  # percent of GDP

master["Extractive_NR_Rents"] = (
    master[RENT_COL].fillna(0)
    - master[FOREST_COL].fillna(0)
).clip(lower=0)

# Countries where extractive rents >= threshold in 1995
rents_1995 = master[master["Year"] == 1995].dropna(subset=[RENT_COL])
resource_countries = set(
    rents_1995[rents_1995["Extractive_NR_Rents"] >= THRESHOLD]["Country Code"].tolist()
)

# Gulf states guaranteed regardless of threshold
GULF = {"ARE", "BHR", "KWT", "OMN", "QAT", "SAU", "IRQ", "IRN", "YEM"}
resource_countries = resource_countries | GULF

# Exclude non-sovereign territories
not_countries = [
    "HKG", "MAC", "PRI", "VIR", "GUM", "ASM", "CYM", "BMU",
    "GRL", "MAF", "SXM", "CUW", "ABW", "FRO", "MNP", "PYF",
]

# Apply filters
cmaster = master[
    (master["Country Code"].isin(resource_countries))
    & (~master["Country Code"].isin(not_countries))
]

print(f"Extractive NR rents >= {THRESHOLD}% in 1995: {len(resource_countries - GULF)} countries + {len(GULF & resource_countries)} Gulf states")
print(f"Total qualifying: {len(resource_countries)} countries")
print(f"Filtered sample: {cmaster.shape[0]:,} rows, "
      f"{cmaster['Country Code'].nunique()} countries, "
      f"{cmaster['Year'].nunique()} years")

# Show which countries qualified
qualifying_df = rents_1995[rents_1995["Country Code"].isin(resource_countries)][
    ["Country Code", "Country Name", RENT_COL, FOREST_COL, "Extractive_NR_Rents"]
].sort_values("Extractive_NR_Rents", ascending=False)
print(f"\nQualifying countries (1995 extractive rents):")
print(qualifying_df.to_string(index=False))


Extractive NR rents >= 1.0% in 1995: 56 countries + 9 Gulf states
Total qualifying: 65 countries
Filtered sample: 1,625 rows, 65 countries, 25 years

Qualifying countries (1995 extractive rents):
Country Code           Country Name  Total natural resources rents (% of GDP)  Forest rents (% of GDP)  Extractive_NR_Rents
         TKM           Turkmenistan                                 47.669660                 0.000000            47.669660
         AGO                 Angola                                 44.210997                 2.985662            41.225335
         YEM            Yemen, Rep.                                 40.159894                 0.073253            40.086642
         KWT                 Kuwait                                 37.330800                 0.000257            37.330543
         OMN                   Oman                                 31.850124                 0.001873            31.848251
         COG            Congo, Rep.                         

## 1. Variable-Level Missingness

For each indicator, we calculate the percentage of missing observations across the full filtered sample, along with the number of countries and years with at least some data. Variables with high missingness may need to be dropped or handled carefully during imputation.

In [3]:
def analyze_variable_missingness(df):
    """Calculate missingness statistics for each variable in the panel."""
    data_cols = [c for c in df.columns if c not in ['Country Code', 'Country Name', 'Year']]

    results = []
    for col in data_cols:
        valid_data = df[df[col].notna()]
        n_obs = valid_data.shape[0]
        n_missing = df[col].isna().sum()
        pct_missing = (n_missing / len(df)) * 100
        n_countries = valid_data['Country Code'].nunique()
        n_years = valid_data['Year'].dropna().nunique()
        year_range = (f"{int(valid_data['Year'].min())}-{int(valid_data['Year'].max())}"
                      if n_years > 0 else "N/A")

        results.append({
            'Variable': col,
            'Valid Obs': n_obs,
            'Missing': n_missing,
            '% Missing': round(pct_missing, 1),
            'Countries': n_countries,
            'Years': n_years,
            'Year Range': year_range,
        })

    return pd.DataFrame(results).sort_values('% Missing', ascending=False)


var_missing = analyze_variable_missingness(cmaster)

print("=" * 80)
print("VARIABLE-LEVEL MISSINGNESS SUMMARY")
print("=" * 80)
print(f"\nTotal observations: {len(cmaster)}")
print(f"Total countries: {cmaster['Country Code'].nunique()}")
print(f"Total years: {cmaster['Year'].dropna().nunique()}")
print()
print(var_missing.to_string(index=False))

# Coverage buckets
print("\n--- Variable Coverage Buckets ---")
print(f"< 10% missing: {(var_missing['% Missing'] < 10).sum()}")
print(f"10-30% missing: {((var_missing['% Missing'] >= 10) & (var_missing['% Missing'] < 30)).sum()}")
print(f"30-50% missing: {((var_missing['% Missing'] >= 30) & (var_missing['% Missing'] < 50)).sum()}")
print(f"> 50% missing: {(var_missing['% Missing'] >= 50).sum()}")

VARIABLE-LEVEL MISSINGNESS SUMMARY

Total observations: 1625
Total countries: 65
Total years: 25

                                                           Variable  Valid Obs  Missing  % Missing  Countries  Years Year Range
                                                  Production_Others         24     1601       98.5          4      6  2014-2019
                                                  Production_Metals        431     1194       73.5         24     25  1995-2019
                              General government structural balance        562     1063       65.4         27     25  1995-2019
                                                  High-tech exports        608     1017       62.6         61     13  2007-2019
                                        General government net debt        672      953       58.6         32     25  1995-2019
                                                          Civil war        744      881       54.2         62     12  1995-2006
      

## 2. Country-Level Missingness

For each country, we calculate the share of cells (across all variables and years) that are missing. Countries with very high missingness are candidates for exclusion from the sample, as imputation would be unreliable.

In [4]:
def analyze_country_missingness(df):
    """Calculate missingness statistics for each country in the panel."""
    data_cols = [c for c in df.columns if c not in ['Country Code', 'Country Name', 'Year']]

    results = []
    for code in df['Country Code'].unique():
        country_data = df[df['Country Code'] == code]
        country_name = country_data['Country Name'].iloc[0]

        n_rows = len(country_data)
        total_cells = n_rows * len(data_cols)
        missing_cells = country_data[data_cols].isna().sum().sum()
        pct_missing = (missing_cells / total_cells) * 100

        complete_vars = sum(country_data[col].notna().all() for col in data_cols)
        vars_with_data = sum(country_data[col].notna().any() for col in data_cols)
        years_covered = country_data['Year'].dropna().nunique()

        results.append({
            'Code': code,
            'Country': country_name,
            'Rows': n_rows,
            'Years Covered': years_covered,
            '% Missing': round(pct_missing, 1),
            'Complete Vars': complete_vars,
            'Vars with Data': vars_with_data,
            'Total Vars': len(data_cols),
        })

    return pd.DataFrame(results).sort_values('% Missing', ascending=False)


country_missing = analyze_country_missingness(cmaster)

print("=" * 80)
print("COUNTRY-LEVEL MISSINGNESS SUMMARY")
print("=" * 80)
print("\nTop 20 countries with MOST missing data:")
print(country_missing.head(20).to_string(index=False))
print("\n\nTop 20 countries with LEAST missing data:")
print(country_missing.tail(20).to_string(index=False))

# Coverage buckets
print("\n--- Country Coverage Buckets ---")
print(f"< 20% missing: {(country_missing['% Missing'] < 20).sum()}")
print(f"20-40% missing: {((country_missing['% Missing'] >= 20) & (country_missing['% Missing'] < 40)).sum()}")
print(f"> 40% missing: {(country_missing['% Missing'] >= 40).sum()}")

COUNTRY-LEVEL MISSINGNESS SUMMARY

Top 20 countries with MOST missing data:
Code                Country  Rows  Years Covered  % Missing  Complete Vars  Vars with Data  Total Vars
 NCL          New Caledonia    25             25       63.3             17              25          61
 BRN      Brunei Darussalam    25             25       35.6             35              41          61
 TKM           Turkmenistan    25             25       29.7             37              45          61
 LBY                  Libya    25             25       29.2             34              48          61
 PNG       Papua New Guinea    25             25       27.1             37              49          61
 GNQ      Equatorial Guinea    25             25       26.3             35              49          61
 SYR   Syrian Arab Republic    25             25       25.0             39              50          61
 SUR               Suriname    25             25       23.3             41              52          

## 5. Descriptive Statistics: High- vs Low-Missingness Countries

To check whether dropping high-missingness countries introduces systematic bias, we compare means of key variables between the five most data-poor countries and the rest of the sample.

In [5]:
vars_of_interest = [
    'GDP per capita (constant prices, PPP)',
    'Total natural resources rents (% of GDP)',
    'Rule of law index',
    'Employment in industry (% of total employment)',
    'Economic Complexity Index',
]

top5_missing_codes = country_missing.head(5)['Code'].tolist()
top5_df = cmaster[cmaster['Country Code'].isin(top5_missing_codes)]
rest_df = cmaster[~cmaster['Country Code'].isin(top5_missing_codes)]

comparison = []
for var in vars_of_interest:
    top5_mean = top5_df[var].mean()
    rest_mean = rest_df[var].mean()
    diff = top5_mean - rest_mean
    pct_diff = (diff / rest_mean * 100) if rest_mean != 0 else np.nan
    comparison.append({
        'Variable': var,
        'Top 5 Missing (Mean)': round(top5_mean, 2),
        'Rest of Countries (Mean)': round(rest_mean, 2),
        'Difference': round(diff, 2),
        '% Difference': round(pct_diff, 1),
    })

comparison_df = pd.DataFrame(comparison)
print("=" * 100)
print("SIDE-BY-SIDE MEAN COMPARISON: Top 5 most data-poor vs rest")
print("=" * 100)
print(comparison_df.to_string(index=False))

SIDE-BY-SIDE MEAN COMPARISON: Top 5 most data-poor vs rest
                                      Variable  Top 5 Missing (Mean)  Rest of Countries (Mean)  Difference  % Difference
         GDP per capita (constant prices, PPP)              27516.42                  17721.28     9795.14          55.3
      Total natural resources rents (% of GDP)                 25.44                     14.38       11.07          77.0
                             Rule of law index                  0.23                      0.44       -0.21         -48.0
Employment in industry (% of total employment)                 19.85                     21.77       -1.92          -8.8
                     Economic Complexity Index                 -1.24                     -0.37       -0.86         230.0


---

## Summary

This diagnostic step identified:
- Which variables have structurally poor coverage (candidates for exclusion or careful imputation)
- Which countries have excessive missingness (candidates for omission from the final sample)
- Whether missingness is concentrated in particular time periods
- Which variable pairs are highly correlated (informing variable selection to avoid multicollinearity)

The findings from this notebook directly inform the country and variable selections applied in Step 3 (`3_Imputing.ipynb`).

In [6]:
# ── NB2 SUMMARY ──
print("=" * 70)
print("NB2: MISSINGNESS DIAGNOSTICS SUMMARY")
print("=" * 70)
print(f"\nSample: {cmaster['Country Code'].nunique()} countries, {cmaster['Year'].nunique()} years, {len(cmaster):,} rows")

print(f"\n--- Variable missingness ---")
for _, row in var_missing.iterrows():
    flag = "***" if row['% Missing'] > 40 else "**" if row['% Missing'] > 20 else ""
    print(f"  {row['Variable']:55s} {row['% Missing']:5.1f}% missing  {flag}")

print(f"\n--- Country missingness (top 15) ---")
for _, row in country_missing.head(15).iterrows():
    print(f"  {row['Code']} {str(row['Country'])[:30]:30s} {row['% Missing']:5.1f}%")


NB2: MISSINGNESS DIAGNOSTICS SUMMARY

Sample: 65 countries, 25 years, 1,625 rows

--- Variable missingness ---
  Production_Others                                        98.5% missing  ***
  Production_Metals                                        73.5% missing  ***
  General government structural balance                    65.4% missing  ***
  High-tech exports                                        62.6% missing  ***
  General government net debt                              58.6% missing  ***
  Civil war                                                54.2% missing  ***
  Welfare-relevant TFP                                     36.9% missing  **
  TFP level (constant national prices)                     36.9% missing  **
  Use of IMF credit (DOD, current US$)                     33.7% missing  **
  Lending interest rate (%)                                33.5% missing  **
  Real interest rate (%)                                   33.5% missing  **
  Reserves_Hydrocarbons             